# Cone Image Detector Pipeline

End-to-end safety-cone detection pipeline combining **VGGT 3-D reconstruction** with **YOLOv8** image-space detection.

**Steps:**
1. Install Requirements
2. Import Libraries & Configure Logging
3. Configure Pipeline Parameters
4. Load VGGT Predictions
5. Render Frames for Detection
6. Run YOLOv8 Cone Detection
7. Map Detections to Source Frames
8. Fit Ground Plane via RANSAC
9. Back-Project 2D Boxes → 3D Points
10. Measure Cone Geometry in 3D
11. Export Results (PLY + JSON/CSV)
12. Annotate & Display Results

## 1. Install Requirements

Install all pipeline dependencies. If a `requirements.txt` is present in the working directory it is used; otherwise packages are installed directly.

In [ ]:
import subprocess, sys, pathlib

# --- Install from requirements.txt if present ---
for candidate in [
    pathlib.Path("/kaggle/working/requirements.txt"),
    pathlib.Path("requirements.txt"),
]:
    if candidate.exists():
        print(f"Installing from {candidate}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(candidate)])
        break

# --- Core pipeline dependencies ---
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "ultralytics",          # YOLOv8
    "open3d",               # 3-D I/O and RANSAC plane fitting
    "scipy",                # ConvexHull for footprint area
    "Pillow",               # image helpers
    "tqdm",                 # progress bars
    "onnxruntime",          # optional: sky segmentation in vggt tools
])
print("Core dependencies installed.")

In [ ]:
# --- OpenCV headless (no GUI window deps) ---
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "opencv-python-headless",
])

# --- VGGT from GitHub (Meta's 3-D reconstruction model) ---
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "git+https://github.com/facebookresearch/vggt.git",
])

# --- viser (optional – interactive 3-D visualisation) ---
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "viser"])
    print("viser installed.")
except Exception:
    print("viser skipped (optional).")

print("All dependencies ready.")

## 2. Import Libraries and Configure Logging

Import all required modules and define `HAS_*` availability flags for optional heavy dependencies.

In [ ]:
from __future__ import annotations

import csv
import json
import logging
import pickle
import sys
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np

# ── matplotlib inline for Kaggle / Jupyter ───────────────────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ── Optional heavy dependencies (graceful fallback if missing) ────────────────
try:
    from PIL import Image as PILImage
    HAS_PIL = True
except ImportError:
    HAS_PIL = False
    print("PIL not found – some image helpers disabled")

try:
    import open3d as o3d
    HAS_O3D = True
except ImportError:
    HAS_O3D = False
    print("open3d not found – using PCA fallback for plane fitting")

try:
    from ultralytics import YOLO
    HAS_YOLO = True
except ImportError:
    HAS_YOLO = False
    print("ultralytics not found – YOLO detection unavailable")

try:
    from scipy.spatial import ConvexHull
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False
    print("scipy not found – footprint area uses bounding-box fallback")

try:
    import torch
    from vggt.models.vggt import VGGT
    from vggt.utils.load_fn import load_and_preprocess_images
    from vggt.utils.pose_enc import pose_encoding_to_extri_intri
    HAS_VGGT = True
except ImportError:
    HAS_VGGT = False
    print("vggt not found – video reconstruction step unavailable")

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
log = logging.getLogger(__name__)

# ── Add the cone-detector directory to sys.path so we can import the module ───
SCRIPT_DIR = Path("/kaggle/working")
for candidate in [Path("."), Path("/kaggle/working"), Path(__file__).parent if "__file__" in dir() else Path(".")]:
    if (candidate / "cone_image_detector.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        SCRIPT_DIR = candidate
        break

# Import the full pipeline module
from cone_image_detector import (
    ConeDetection2D, ConeDetection3D,
    _fit_ground_ransac, _height_above_plane, _footprint_area,
    load_predictions,
    select_best_frame, render_frame_image, render_composite_image,
    load_yolo_model, detect_cones_yolo, map_detections_to_frames,
    backproject_box_to_3d, measure_cone_3d,
    annotate_image, save_cone_ply, save_summary,
    vggt_reconstruct_from_video,
)

print(f"Module loaded from: {SCRIPT_DIR}")
print(f"  HAS_PIL={HAS_PIL}  HAS_O3D={HAS_O3D}  HAS_YOLO={HAS_YOLO}  HAS_SCIPY={HAS_SCIPY}  HAS_VGGT={HAS_VGGT}")

## 3. Configure Pipeline Parameters

Edit the variables below to match your input data and desired output settings.

In [ ]:
# ── Input ─────────────────────────────────────────────────────────────────────
# Set INPUT_MODE to one of: "pred_pkl" | "pred_dir" | "video"
INPUT_MODE      = "pred_pkl"

# Path to a pre-computed VGGT .pkl prediction file (used when INPUT_MODE = "pred_pkl")
PRED_PKL_PATH   = "/kaggle/input/vggt-predictions/predictions.pkl"

# Directory containing prediction files (used when INPUT_MODE = "pred_dir")
PRED_DIR        = "/kaggle/input/vggt-predictions/"

# Video file to run end-to-end reconstruction on (used when INPUT_MODE = "video")
VIDEO_PATH      = "/kaggle/input/site-footage/site.mp4"

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_DIR      = "/kaggle/working/cone_results"

# ── VGGT reconstruction settings (only relevant for INPUT_MODE = "video") ─────
MAX_FRAMES      = 60       # max video frames to pass to VGGT
TARGET_FPS      = 2.0      # sub-sample video to ~this fps before VGGT
VGGT_MODEL_PATH = None     # local .pt weights; None = auto-download from HuggingFace
DEVICE          = None     # "cuda" | "cpu" | None (auto-detect)
SAVE_PRED_PKL   = True     # persist predictions.pkl so VGGT can be skipped next run

# ── Detection thresholds ──────────────────────────────────────────────────────
YOLO_MODEL_PATH       = "yolov8n.pt"   # swap for a cone-specific checkpoint for best accuracy
CONF_THRESHOLD        = 0.35
IOU_THRESHOLD         = 0.45
DEPTH_CONF_THRESHOLD  = 0.30

# ── Pipeline options ──────────────────────────────────────────────────────────
USE_COMPOSITE   = True     # tile all frames into one image (True) or use best frame only (False)
SAVE_PLY        = True     # write per-cone .ply point cloud files

print("Configuration:")
for k, v in {
    "INPUT_MODE": INPUT_MODE, "OUTPUT_DIR": OUTPUT_DIR,
    "YOLO_MODEL": YOLO_MODEL_PATH, "CONF": CONF_THRESHOLD,
    "USE_COMPOSITE": USE_COMPOSITE,
}.items():
    print(f"  {k:20s} = {v}")

## 4. Load VGGT Predictions

Load the VGGT prediction dict from a `.pkl` file, a directory, or run end-to-end reconstruction from a raw video (Step 0).

In [ ]:
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

if INPUT_MODE == "video":
    # ── Step 0: Run VGGT reconstruction from raw video ────────────────────────
    print(f"Running VGGT reconstruction on: {VIDEO_PATH}")
    pred = vggt_reconstruct_from_video(
        video_path=VIDEO_PATH,
        output_dir=OUTPUT_DIR,
        max_frames=MAX_FRAMES,
        target_fps=TARGET_FPS if TARGET_FPS and TARGET_FPS > 0 else None,
        device=DEVICE,
        vggt_model_path=VGGT_MODEL_PATH,
        save_predictions=SAVE_PRED_PKL,
    )
    print("VGGT reconstruction complete.")
else:
    # ── Load pre-computed predictions ─────────────────────────────────────────
    src = PRED_PKL_PATH if INPUT_MODE == "pred_pkl" else PRED_DIR
    print(f"Loading predictions from: {src}")
    pred = load_predictions(src)
    print("Predictions loaded.")

# ── Print shapes of all keys ──────────────────────────────────────────────────
print("\nPrediction dict shapes:")
for key, val in pred.items():
    if hasattr(val, "shape"):
        print(f"  {key:25s}: {val.shape}  dtype={val.dtype}")
    else:
        print(f"  {key:25s}: {type(val).__name__}")

## 5. Render Frames for Detection

Produce a single image for YOLO inference. With `USE_COMPOSITE=True` all frames are tiled into one image; with `False` only the best-confidence frame is used.

In [ ]:
if USE_COMPOSITE:
    detect_image, tile_info = render_composite_image(pred)
    print(f"Composite image: {detect_image.shape[1]}×{detect_image.shape[0]} px  ({len(tile_info)} frame tiles)")
else:
    best_frame_idx = select_best_frame(pred)
    detect_image = render_frame_image(pred, best_frame_idx)
    tile_info = [dict(frame_idx=best_frame_idx, row=0, col=0,
                      h=detect_image.shape[0], w=detect_image.shape[1])]
    print(f"Single frame {best_frame_idx}: {detect_image.shape[1]}×{detect_image.shape[0]} px")

# Save render to disk
render_path = Path(OUTPUT_DIR) / "render_for_detection.png"
cv2.imwrite(str(render_path), detect_image)
print(f"Saved render to: {render_path}")

# Display inline (convert BGR → RGB for matplotlib)
plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(detect_image, cv2.COLOR_BGR2RGB))
plt.title("Input Image for YOLO Detection")
plt.axis("off")
plt.tight_layout()
plt.savefig(str(Path(OUTPUT_DIR) / "render_preview.png"), dpi=80)
plt.show()
print("Render preview saved.")

## 6. Run YOLOv8 Cone Detection

Load the YOLO model and run inference on the rendered image. The model accepts any class whose name contains `"cone"`, `"traffic"`, or `"barrier"`. For best accuracy, use a fine-tuned cone-detection checkpoint.

In [ ]:
yolo_model = load_yolo_model(YOLO_MODEL_PATH)

raw_detections = detect_cones_yolo(
    yolo_model,
    detect_image,
    conf_threshold=CONF_THRESHOLD,
    iou_threshold=IOU_THRESHOLD,
)

print(f"\nRaw detections ({len(raw_detections)} total):")
for i, d in enumerate(raw_detections):
    print(f"  [{i}] label={d['label']:20s}  conf={d['conf']:.3f}"
          f"  box=({d['x1']:.0f},{d['y1']:.0f})-({d['x2']:.0f},{d['y2']:.0f})")

if not raw_detections:
    print("\n⚠ No detections – try lowering CONF_THRESHOLD or use a cone-specific YOLO model.")

## 7. Map Detections to Source Frames

Translate bounding-box coordinates from the composite image back to per-frame pixel coordinates. Each detection is stored as a `ConeDetection2D` dataclass.

In [ ]:
dets_2d: List[ConeDetection2D] = map_detections_to_frames(raw_detections, tile_info)

print(f"Mapped {len(dets_2d)} detection(s) to source frames:\n")
for d in dets_2d:
    print(f"  frame={d.frame_idx}  label={d.label:20s}  conf={d.conf:.3f}"
          f"  box=({d.x1:.0f},{d.y1:.0f})-({d.x2:.0f},{d.y2:.0f})")

# Save annotated 2-D detections image
ann_2d = annotate_image(detect_image, dets_2d, [], tile_info)
ann_2d_path = Path(OUTPUT_DIR) / "detections_2d.png"
cv2.imwrite(str(ann_2d_path), ann_2d)
print(f"\nSaved 2-D annotated image → {ann_2d_path}")

# Display inline
plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(ann_2d, cv2.COLOR_BGR2RGB))
plt.title(f"2D Detections ({len(dets_2d)} boxes)")
plt.axis("off")
plt.tight_layout()
plt.show()

## 8. Fit Ground Plane via RANSAC

Fit a robust ground plane to the best frame's world-space point cloud using RANSAC (Open3D) or PCA fallback. The `ground_normal` and scalar `d` define the plane equation `normal · p + d = 0`.

In [ ]:
# Use the best-confidence frame for ground plane estimation
best_frame_idx = tile_info[0]["frame_idx"]
wp_best = pred["world_points"][best_frame_idx].reshape(-1, 3)
finite_mask = np.all(np.isfinite(wp_best), axis=1) & np.any(wp_best != 0, axis=1)
wp_valid = wp_best[finite_mask]

ground_normal, ground_d = _fit_ground_ransac(wp_valid)

print(f"Ground plane  normal = {ground_normal.round(4)}")
print(f"              d      = {ground_d:.4f}")
print(f"              (plane equation: {ground_normal[0]:.3f}·x + {ground_normal[1]:.3f}·y + {ground_normal[2]:.3f}·z + {ground_d:.3f} = 0)")

# Visualise point cloud coloured by height above the ground plane
heights = _height_above_plane(wp_valid, ground_normal, ground_d)
above_ground = wp_valid[heights > 0.02]   # filter out below-ground noise
h_vals = heights[heights > 0.02]

if len(above_ground) > 0:
    sample = np.random.choice(len(above_ground), min(20_000, len(above_ground)), replace=False)
    fig = plt.figure(figsize=(10, 5))
    ax = fig.add_subplot(111, projection="3d")
    sc = ax.scatter(
        above_ground[sample, 0], above_ground[sample, 1], above_ground[sample, 2],
        c=h_vals[sample], cmap="plasma", s=0.5, alpha=0.6,
    )
    plt.colorbar(sc, ax=ax, label="Height above ground (m)", shrink=0.6)
    ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")
    ax.set_title("Scene Point Cloud – Coloured by Height above Ground")
    plt.tight_layout()
    plt.savefig(str(Path(OUTPUT_DIR) / "ground_plane_viz.png"), dpi=80)
    plt.show()
    print("Height-coloured scatter plot saved.")

## 9. Back-Project 2D Boxes → 3D Points

For each 2-D bounding box, extract the corresponding 3-D world-space points from the VGGT `world_points` map, applying a confidence threshold to filter low-quality depth estimates.

In [ ]:
box_clusters: List[Tuple] = []   # (det, xyz, rgb) per detection

print(f"Back-projecting {len(dets_2d)} detection(s) to 3-D:\n")
for det in dets_2d:
    xyz, rgb = backproject_box_to_3d(pred, det, depth_conf_threshold=DEPTH_CONF_THRESHOLD)
    box_clusters.append((det, xyz, rgb))
    print(f"  frame={det.frame_idx}  label={det.label:18s}  → {len(xyz):5d} valid 3-D pts")

# Quick scatter visualisation of extracted clusters
if box_clusters:
    fig = plt.figure(figsize=(10, 5))
    ax = fig.add_subplot(111, projection="3d")
    palette = plt.cm.tab10.colors
    for i, (det, xyz, rgb) in enumerate(box_clusters):
        if len(xyz) == 0:
            continue
        color = palette[i % len(palette)]
        ax.scatter(xyz[:, 0], xyz[:, 1], xyz[:, 2],
                   c=[color], s=1.5, alpha=0.7,
                   label=f"{det.label} (fr{det.frame_idx})")
    ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")
    ax.set_title("Back-Projected 3-D Clusters per Detection")
    ax.legend(fontsize=7, markerscale=4)
    plt.tight_layout()
    plt.savefig(str(Path(OUTPUT_DIR) / "backproject_clusters.png"), dpi=80)
    plt.show()

## 10. Measure Cone Geometry in 3D

For each back-projected cluster, compute:
- **height_m** – height above the RANSAC ground plane
- **footprint_area_m²** – convex-hull area on the ground plane
- **aspect_ratio** – height / √footprint (cones typically > 2)
- **score** – composite quality score (0–1)

In [ ]:
cone_clusters: List[ConeDetection3D] = []

for det, xyz, rgb in box_clusters:
    cone = measure_cone_3d(xyz, rgb, det, ground_normal, ground_d)
    if cone is not None:
        cone_clusters.append(cone)

print(f"\nMeasured {len(cone_clusters)} cone(s):\n")
print(f"{'#':>3}  {'frame':>5}  {'label':18}  {'conf':>5}  {'height_m':>8}  {'foot_m²':>8}  {'aspect':>7}  {'score':>6}  {'centroid'}")
print("─" * 100)
for i, c in enumerate(cone_clusters):
    cx, cy, cz = c.centroid
    print(
        f"{i:>3}  {c.det2d.frame_idx:>5}  {c.det2d.label:18}  {c.det2d.conf:>5.3f}"
        f"  {c.height_m:>8.3f}  {c.footprint_area_m2:>8.4f}  {c.aspect_ratio:>7.2f}"
        f"  {c.score:>6.3f}  ({cx:.2f}, {cy:.2f}, {cz:.2f})"
    )

if not cone_clusters:
    print("No cones measured – clusters may have been too small or too flat.")

## 11. Export Results (PLY + JSON/CSV)

Save each detected cone as a coloured `.ply` point cloud, and write a `cone_summary.json` + `cone_summary.csv` with all measurements.

In [ ]:
out_dir = Path(OUTPUT_DIR)

# ── Save per-cone PLY files ────────────────────────────────────────────────────
if SAVE_PLY and cone_clusters:
    print("Saving PLY files:")
    for i, cone in enumerate(cone_clusters):
        ply_path = save_cone_ply(cone, out_dir, i)
        print(f"  [{i}] {ply_path.name}  ({len(cone.points_xyz)} pts)")
else:
    print("PLY export skipped (SAVE_PLY=False or no cones detected).")

# ── Save JSON / CSV summary ────────────────────────────────────────────────────
save_summary(cone_clusters, out_dir)

# ── List all output files ─────────────────────────────────────────────────────
print(f"\nAll files in {OUTPUT_DIR}:")
for f in sorted(out_dir.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:45s}  {size_kb:8.1f} KB")

# ── Preview the JSON summary ──────────────────────────────────────────────────
json_path = out_dir / "cone_summary.json"
if json_path.exists():
    with open(json_path) as jf:
        summary_data = json.load(jf)
    print(f"\ncone_summary.json ({len(summary_data)} records):")
    print(json.dumps(summary_data[:3], indent=2))   # show first 3 records

# ── Display CSV with pandas (optional) ───────────────────────────────────────
csv_path = out_dir / "cone_summary.csv"
if csv_path.exists():
    try:
        import pandas as pd
        df = pd.read_csv(csv_path)
        print(f"\ncone_summary.csv:")
        display(df)
    except ImportError:
        print("pandas not available – skipping CSV display.")

## 12. Annotate & Display Results

Draw bounding boxes with 3-D measurements on the detection image and display both the 2-D annotated image and individual cropped cone thumbnails.

In [ ]:
# ── Annotated composite image with 3-D labels ─────────────────────────────────
ann_3d = annotate_image(detect_image, dets_2d, cone_clusters, tile_info)
ann_3d_path = out_dir / "detections_3d.png"
cv2.imwrite(str(ann_3d_path), ann_3d)
print(f"Saved 3-D annotated image → {ann_3d_path}")

plt.figure(figsize=(16, 9))
plt.imshow(cv2.cvtColor(ann_3d, cv2.COLOR_BGR2RGB))
plt.title(f"Final Annotated Image – {len(cone_clusters)} Cone(s) Detected")
plt.axis("off")
plt.tight_layout()
plt.show()

# ── Crop thumbnails for each detection ────────────────────────────────────────
if dets_2d:
    n = len(dets_2d)
    n_cols = min(4, n)
    n_rows = int(np.ceil(n / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
    axes = np.array(axes).reshape(-1)

    img_rgb = cv2.cvtColor(detect_image, cv2.COLOR_BGR2RGB)
    for idx, det in enumerate(dets_2d):
        # Map back to composite image coords
        ox, oy = 0, 0
        if tile_info:
            for tile in tile_info:
                if tile["frame_idx"] == det.frame_idx:
                    ox, oy = tile["col"] * tile["w"], tile["row"] * tile["h"]
                    break
        x1, y1 = max(0, int(det.x1 + ox) - 5), max(0, int(det.y1 + oy) - 5)
        x2, y2 = min(img_rgb.shape[1], int(det.x2 + ox) + 5), min(img_rgb.shape[0], int(det.y2 + oy) + 5)
        crop = img_rgb[y1:y2, x1:x2]
        axes[idx].imshow(crop)

        # Pull 3-D metrics if available
        cone = next((c for c in cone_clusters if c.det2d is det), None)
        label = f"fr{det.frame_idx} {det.label}\nconf={det.conf:.2f}"
        if cone:
            label += f"\nh={cone.height_m:.2f}m  sc={cone.score:.2f}"
        axes[idx].set_title(label, fontsize=8)
        axes[idx].axis("off")

    for ax in axes[n:]:
        ax.set_visible(False)

    plt.suptitle("Detection Thumbnails", fontsize=12)
    plt.tight_layout()
    plt.savefig(str(out_dir / "thumbnails.png"), dpi=80)
    plt.show()

# ── Optional Open3D 3-D viewer (disabled in headless environments) ─────────────
OPEN_3D_VIEWER = False   # set to True in a local interactive session

if OPEN_3D_VIEWER and HAS_O3D and cone_clusters:
    from cone_image_detector import visualize_open3d
    visualize_open3d(pred, cone_clusters, frame_idx=best_frame_idx)

# ── Final summary ─────────────────────────────────────────────────────────────
print(f"\n{'═'*60}")
print(f"  DETECTED {len(cone_clusters)} SAFETY CONE(S)")
print(f"{'═'*60}")
for i, c in enumerate(cone_clusters):
    print(f"  [{i:02d}] h={c.height_m:.2f}m  foot={c.footprint_area_m2:.4f}m²  score={c.score:.3f}  @ {c.centroid.round(2)}")
print(f"\nAll results saved to: {OUTPUT_DIR}/")